# Naive Bayes Classifier

This notebook contains a brief explanation and derivation of **Bayes' Theorem**, followed by a **Gaussian Naive Bayes** classifier implemented from scratch in Python. It finishes with an example evaluation on the Iris dataset.

## Bayes' Theorem — Brief Description

Bayes' theorem relates conditional probabilities. It lets you update the probability estimate for a hypothesis given new evidence.

Mathematically:
$P(A\mid B) = \dfrac{P(B\mid A) \, P(A)}{P(B)}$

- **P(A|B)**: Posterior probability — probability of hypothesis A given data B.
- **P(B|A)**: Likelihood — probability of observing data B given hypothesis A.
- **P(A)**: Prior probability of hypothesis A.
- **P(B)**: Marginal probability of data B.

## Derivation

Start from the definition of conditional probability:

$P(A\mid B) = \dfrac{P(A \cap B)}{P(B)}$ and $P(B\mid A) = \dfrac{P(A \cap B)}{P(A)}$.
Rearrange the second equation to get $P(A \cap B) = P(B\mid A) P(A)$ and substitute into the first to obtain Bayes' theorem:

$P(A\mid B) = \dfrac{P(B\mid A) P(A)}{P(B)}$.

## Gaussian Naive Bayes — Implementation

This implementation assumes continuous features that follow a Gaussian (normal) distribution conditioned on the class.

In [ ]:

import numpy as np


class GaussianNaiveBayes:
    """Simple Gaussian Naive Bayes implemented from scratch."""
    def __init__(self, var_smoothing=1e-6):
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.class_count_ = {}
        self.class_prior_ = {}
        self.theta_ = {}  # mean
        self.sigma_ = {}  # variance

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_, counts = np.unique(y, return_counts=True)
        self.class_count_ = dict(zip(self.classes_, counts, strict=False))
        n_samples = X.shape[0]
        for cls in self.classes_:
            X_c = X[y == cls]
            self.theta_[cls] = X_c.mean(axis=0)
            # population variance (ddof=0)
            self.sigma_[cls] = X_c.var(axis=0) + self.var_smoothing
            self.class_prior_[cls] = X_c.shape[0] / n_samples
        return self

    def _gaussian_log_prob(self, x, mean, var):
        # Compute log of Gaussian PDF for each feature (to avoid underflow)
        # log(1/sqrt(2*pi*var)) - (x-mean)^2 / (2*var)
        return -0.5 * (np.log(2.0 * np.pi * var) + ((x - mean) ** 2) / var)

    def _joint_log_likelihood(self, x):
        # For each class compute log prior + sum of log likelihoods
        jll = {}
        for cls in self.classes_:
            prior = np.log(self.class_prior_[cls])
            ll = self._gaussian_log_prob(x, self.theta_[cls], self.sigma_[cls]).sum()
            jll[cls] = prior + ll
        return jll

    def predict(self, X):
        X = np.asarray(X)
        y_pred = []
        for x in X:
            jll = self._joint_log_likelihood(x)
            # choose class with highest joint log likelihood
            pred = max(jll.items(), key=lambda kv: kv[1])[0]
            y_pred.append(pred)
        return np.array(y_pred)

    def predict_proba(self, X):
        X = np.asarray(X)
        probas = []
        for x in X:
            jll = self._joint_log_likelihood(x)
            # convert log-likelihoods to probabilities safely
            vals = np.array(list(jll.values()))
            # subtract max for numerical stability
            shifted = vals - vals.max()
            exp_vals = np.exp(shifted)
            probs = exp_vals / exp_vals.sum()
            # return in same class order
            probas.append(dict(zip(list(jll.keys()), probs, strict=False)))
        return probas


In [ ]:
# Example: evaluate on Iris dataset
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

nb = GaussianNaiveBayes()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))


## Notes / Next steps

- This notebook implements **Gaussian Naive Bayes**. For **text** data consider implementing **Multinomial Naive Bayes** (counts) or **Bernoulli Naive Bayes** (binary features).
- You can extend this by adding Laplace smoothing, handling categorical features, or vectorizing computations for speed.